In [3]:
import json
import re
import pandas as pd
from typing import Dict, List, Tuple
import html 
from Bio.SeqUtils import molecular_weight
import time
from rapidfuzz.distance import Levenshtein
import argparse
from Bio import Entrez
from concurrent.futures import ThreadPoolExecutor, as_completed
import logging


In [9]:
df = pd.read_csv('BaAMP_data.csv')
df.head(5)

,Source_URL,Accession,Activity,Date,Administration,Model,Organism,Status,Notes,Sequence,Peptide_Name,Exp_Method,Exp_Concentration,Exp_Notes
0,https://www.baamps.it/experiment/1,EXP00001,Formation (3-24h),2013-11-08,In solution,in vitro,Staphylococcus aureus,Validated,Sa4 clinical isolate,RGLRRLGRKIAHGVKKYGPTVLRIIRIA,SMAP-29,Crystal violet assay,0.626 uM,No notes
1,https://www.baamps.it/experiment/2,EXP00002,Formation (3-24h),2013-11-08,In solution,in vitro,Staphylococcus aureus,Validated,Sa4 clinical isolate,GGLRSLGRKILRAWKKYGPIIVPIIRI,BMAP-28,Crystal violet assay,2.603 uM,No notes
2,https://www.baamps.it/experiment/3,EXP00003,Formation (3-24h),2013-11-08,In solution,in vitro,Staphylococcus aureus,Validated,Sa7 clinical isolate,GRFKRFRKKFKKLFKKLSPVIPLLHL,BMAP-27,Crystal violet assay,2.481 uM,No notes
3,https://www.baamps.it/experiment/4,EXP00004,Formation (3-24h),2013-11-08,In solution,in vitro,Stenotrophomonas maltophilia,Validated,Sm143 clinical isolate,GRFKRFRKKFKKLFKKLSPVIPLLHL,BMAP-27,Crystal violet assay,1.241 uM,No notes
4,https://www.baamps.it/experiment/5,EXP00005,Formation (3-24h),2013-11-08,In solution,in vitro,Stenotrophomonas maltophilia,Validated,Sm109 clinical isolate,RGLRRLGRKIAHGVKKYGPTVLRIIRIA,SMAP-29,Crystal violet assay,0.626 uM,Sm109 strain


In [10]:
# 配置日志
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# 保留Accession(BAAMP_ID)、Sequence、Length、MIC_Value(ug/ml)、Tax_ID、Raw_Species、Target_Species
# 清洗序列
print(f"初始数据集中存在的行总共有{len(df)}\n")
# 氨基酸序列要清洗，保留20种标准的氨基酸字母
def clean_sequence(seq: str) -> str:
    seq = seq.upper().strip()
    standard_aa = set("ACDEFGHIKLMNPQRSTVWY")
    if set(seq).issubset(standard_aa):
        return seq
    else:
        return None
df["Sequence"] = df["Sequence"].apply(clean_sequence)
df = df.dropna(subset=['Sequence'])
print(f"氨基酸序列清洗后存在的行总共有{len(df)}\n")

df["ID"] = "BA" + df["Accession"].str.strip("EXP")
df["Length"] = df["Sequence"].apply(lambda x : len(str(x)))

def unify_to_ug_ml(row):
    """
    统一MIC值单位为μg/mL
    """
    raw_val = str(row['Exp_Concentration']).lower()
    seq = row['Sequence']
    
    try:
        # 1. 提取数值
        match = re.search(r'([\d\.]+)', raw_val)
        if not match: 
            return None
        value = float(match.group(1))

        # 2. 如果已经是 ug/ml，直接返回数值
        if 'ug/ml' in raw_val:
            return value
            
        # 3. 如果是 uM，则根据序列计算分子量并转换
        if 'um' in raw_val or 'uM' in raw_val:
            # 计算分子量 (Da)
            mw = molecular_weight(seq, seq_type="protein")
            # 转换公式：(uM * MW) / 1000 = ug/ml
            return (value * mw) / 1000
        
        return value # 默认返回
    except Exception as e:
        logger.warning(f"MIC值转换失败 ({raw_val}): {e}")
        return None

df["MIC_Value(ug/ml)"] = df.apply(unify_to_ug_ml, axis=1)
df.rename(columns={"Organism": "Raw_Species"}, inplace=True)
df.dropna(subset=['MIC_Value(ug/ml)'], inplace=True)
df.drop_duplicates(inplace=True)
print(df.head(5))


初始数据集中存在的行总共有560

氨基酸序列清洗后存在的行总共有560

                           Source_URL Accession           Activity  \
0  https://www.baamps.it/experiment/1  EXP00001  Formation (3-24h)   
1  https://www.baamps.it/experiment/2  EXP00002  Formation (3-24h)   
2  https://www.baamps.it/experiment/3  EXP00003  Formation (3-24h)   
3  https://www.baamps.it/experiment/4  EXP00004  Formation (3-24h)   
4  https://www.baamps.it/experiment/5  EXP00005  Formation (3-24h)   

         Date Administration     Model                   Raw_Species  \
0  2013-11-08    In solution  in vitro         Staphylococcus aureus   
1  2013-11-08    In solution  in vitro         Staphylococcus aureus   
2  2013-11-08    In solution  in vitro         Staphylococcus aureus   
3  2013-11-08    In solution  in vitro  Stenotrophomonas maltophilia   
4  2013-11-08    In solution  in vitro  Stenotrophomonas maltophilia   

      Status                   Notes                      Sequence  \
0  Validated    Sa4 clinical isolate  

In [11]:
df["Target_Species"] = ""
df["Tax_ID"] = ""
df = df[['ID', 'Sequence', 'Length', 'MIC_Value(ug/ml)', 'Tax_ID',  'Raw_Species', 'Target_Species']]
df.to_csv('BaAMP_MIC_processed.csv', index=False)